In [1]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False



Welcome to JupyROOT 6.18/00


In [2]:
def remap_histograms(hists):
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY_jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W_jets"] = ("W+*",)
#     bkg_map["W+HF"] = ("W+HF",)
#     bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z_jets"] = ("Z+*",)
#     bkg_map["Z+HF"] = ("Z+HF",)
#     bkg_map["Z+LF"] = ("Z+LF",)
#     bkg_map["G+jets"] = ("G+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SingleMuon"] = ("SingleMuon",)
    data_map["EGamma"] = ("EGamma",)
    sig_map["Mphi1995_Mchi1000"] = ("Mphi-1995_Mchi-1000")
    sig_map["Mphi2000_Mchi1500"] = ("Mphi-2000_Mchi-1500")
    sig_map["Mphi2000_Mchi500"] = ("Mphi-2000_Mchi-500")
#     haven't processed signal yet
    for signal in hists['sig']['template'].identifiers('process'):
        if 'mchi' not in str(signal): continue
        sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
        signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial mT_corrected binning',bkg_hists["template"].axis("mT_corrected").edges())


    bkg_hists["template"] = bkg_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    signal_hists["template"] = signal_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    data_hists["template"] = data_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )

    hists = {"bkg": bkg_hists, "data": data_hists, "sig":signal_hists}

    return hists


In [ ]:
np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])

In [3]:
year = '2018'

mT_corrected_binning_dict = {
    "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
}
mT_corrected_binning = mT_corrected_binning_dict[year]
mT_correctedbins = np.array(mT_corrected_binning)
nmT_corrected = len(mT_correctedbins) - 1

###
# Extract histograms from input file
###

print("Grouping histograms")
hists = load("../../../hists/UL_lep_monotop_v1_test_" + year + ".scaled")
hists = remap_histograms(hists)
data_hists = hists["data"]
bkg_hists = hists["bkg"]
signal_hists = hists["sig"]

###
# Preparing histograms for fit
##

data = {}
for r in data_hists["template"].identifiers("region"):
    data[str(r)] = data_hists["template"].integrate("region", r)

background = {}
for r in bkg_hists["template"].identifiers("region"):
    background[str(r)] = bkg_hists["template"].integrate("region", r)

signal = {}
for r in signal_hists["template"].identifiers("region"):
    signal[str(r)] = signal_hists["template"].integrate("region", r)
    
    

Grouping histograms
initial mT_corrected binning [100. 125. 150. 165. 180. 200. 220. 240. 270. 310. 350. 500.]


In [4]:
def addBtagSyst(dictionary, process, region, templ):
    btagUp = template(dictionary, process, "btagUp", region)[0]
    btagDown = template(dictionary, process, "btagDown", region)[0]
    print(btagUp)
    templ.setParamEffect(btag, btagUp, btagDown)

def addVJetsSyst(dictionary, process, region, templ):
    def addSyst(dictionary, process, region, templ, syst, string):
        histogram = dictionary[region].integrate("process", process)
#         print(histogram.integrate("systematic", "nominal").sum('npvs').values()[()])
        nominal=histogram.integrate("systematic", "nominal").sum('npvs').values()[()]
        up=histogram.integrate("systematic", string+"Up").sum('npvs').values()[()]
        down=histogram.integrate("systematic",string+"Down").sum('npvs').values()[()]
        systUp = np.array( up.sum() / nominal.sum() )
        systUp[np.isnan(systUp)] = 1.
        systUp = systUp.sum()
        print(systUp)
        templ.setParamEffect(syst, systUp)
    addSyst(dictionary, process, region, templ, ew1, "ew1")
#     addSyst(dictionary, process, region, templ, ew2W, "ew2W")
#     addSyst(dictionary, process, region, templ, ew2Z, "ew2Z")
#     addSyst(dictionary, process, region, templ, ew3W, "ew3W")
#     addSyst(dictionary, process, region, templ, ew3Z, "ew3Z")
#     addSyst(dictionary, process, region, templ, mix, "mix")
#     addSyst(dictionary, process, region, templ, qcd1, "qcd1")
#     addSyst(dictionary, process, region, templ, qcd2, "qcd2")
#     addSyst(dictionary, process, region, templ, qcd3, "qcd3")

In [15]:
background['ttbare'].integrate("process", "W_jets").integrate('systematic', 'ew1Up').sum('npvs').values()

{}

In [16]:
background['ttbare'].integrate("process", "DY_jets").identifiers('systematic')

[<StringBin (btagDown) instance at 0x7f54251f65c0>,
 <StringBin (btagUp) instance at 0x7f54251f6588>,
 <StringBin (ew1Down) instance at 0x7f54251f6eb8>,
 <StringBin (ew1Up) instance at 0x7f54251f6e48>,
 <StringBin (ew2GDown) instance at 0x7f54251f6f60>,
 <StringBin (ew2GUp) instance at 0x7f54251f6f28>,
 <StringBin (ew2WDown) instance at 0x7f542522e048>,
 <StringBin (ew2WUp) instance at 0x7f54251f6fd0>,
 <StringBin (ew2ZDown) instance at 0x7f542522e0f0>,
 <StringBin (ew2ZUp) instance at 0x7f542522e0b8>,
 <StringBin (ew3GDown) instance at 0x7f542522e198>,
 <StringBin (ew3GUp) instance at 0x7f542522e160>,
 <StringBin (ew3WDown) instance at 0x7f542522e240>,
 <StringBin (ew3WUp) instance at 0x7f542522e208>,
 <StringBin (ew3ZDown) instance at 0x7f542522e2e8>,
 <StringBin (ew3ZUp) instance at 0x7f542522e2b0>,
 <StringBin (id_eDown) instance at 0x7f54251f66d8>,
 <StringBin (id_eUp) instance at 0x7f54251f66a0>,
 <StringBin (isolation_eDown) instance at 0x7f54251f67b8>,
 <StringBin (isolation_eU

In [7]:
def template(dictionary, process, systematic, region, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process).sum('npvs')
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    zerobins = nominal <= 0.
    output = nominal
    
    if "nominal" not in systematic and "data" not in systematic:
    #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("mT_corrected")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "mT_corrected", sumw2)
    return (output, binning, "mT_corrected")

In [8]:
###
###
# Setting up systematics
###
###
lumi = rl.NuisanceParameter("lumi" + year, "lnN")
# zjets_norm = rl.NuisanceParameter("zjets_norm", "lnN")
# wjets_norm = rl.NuisanceParameter("wjets_norm", "lnN")
# tt_norm = rl.NuisanceParameter("tt_norm", "lnN")
id_e = rl.NuisanceParameter("id_e" + year, "shape")
# id_mu = rl.NuisanceParameter("id_mu" + year, "lnN")
# id_pho = rl.NuisanceParameter("id_pho" + year, "lnN")
# reco_e = rl.NuisanceParameter("reco_e" + year, "lnN")
# iso_mu = rl.NuisanceParameter("iso_mu" + year, "lnN")
trig_e = rl.NuisanceParameter("trig_e" + year, "shape")
# trig_met = rl.NuisanceParameter("trig_met" + year, "lnN")
# trig_pho = rl.NuisanceParameter("trig_pho" + year, "lnN")
# veto_tau = rl.NuisanceParameter("veto_tau" + year, "lnN")
# jec = rl.NuisanceParameter("jec" + year, "lnN")
# btag = rl.NuisanceParameter("btag" + year, "shapeN")  # AK4 btag
btag = rl.NuisanceParameter("btag" + year, "shape")  # AK4 btag
ew1 = rl.NuisanceParameter("ew1", "shape")
# #ew2G = rl.NuisanceParameter("ew2G", "lnN")
# ew2W = rl.NuisanceParameter("ew2W", "lnN")
# ew2Z = rl.NuisanceParameter("ew2Z", "lnN")
# #ew3G = rl.NuisanceParameter("ew3G", "lnN")
# ew3W = rl.NuisanceParameter("ew3W", "lnN")
# ew3Z = rl.NuisanceParameter("ew3Z", "lnN")
# mix = rl.NuisanceParameter("mix", "lnN")
# #muF = rl.NuisanceParameter("muF", "lnN")
# #muR = rl.NuisanceParameter("muR", "lnN")
# qcd1 = rl.NuisanceParameter("qcd1", "lnN")
# qcd2 = rl.NuisanceParameter("qcd2", "lnN")
# qcd3 = rl.NuisanceParameter("qcd3", "lnN")
# whf_fraction = rl.NuisanceParameter("whf_fraction", "lnN")
# zhf_fraction = rl.NuisanceParameter("zhf_fraction", "lnN")
#ghf_fraction = rl.NuisanceParameter("ghf_fraction", "shapeN")
pileup = rl.NuisanceParameter("pileup"+year, "shape")

In [9]:
year='2018'
model_id = 'ControlRegionModel'
model = rl.Model(model_id)

ch_name = "ttbare" 
# Create a channel/Control region
ttbare = rl.Channel(ch_name)
# add the Channel/CR to the model
model.addChannel(ttbare)

###
# Add data distribution to the channel
###

if year == "2018":
    ttbare.setObservation(template(data, "EGamma", "data", "ttbare"))
else:
    ttbare.setObservation(template(data, "SingleElectron", "data","ttbare"))

###
# W(->lnu)+jets data-driven model
###

ttbare_wjetsTemplate = template(background, "W_jets", "nominal", "ttbare", read_sumw2=True)
ttbare_wjetsMC = rl.TemplateSample(
    ch_name + "_wjetsMC",
    rl.Sample.BACKGROUND,
    ttbare_wjetsTemplate
)
ttbare_wjetsMC.setParamEffect(lumi, 1.027)
# ttbare_wjetsMC.setParamEffect(trig_e, 1.01)
# ttbare_wjetsMC.setParamEffect(veto_tau, 1.03)
# ttbare_wjetsMC.setParamEffect(wjets_norm, 1.4)
# ttbare_wjetsMC.setParamEffect(jec, 1.05)
# ttbare_wjetsMC.setParamEffect(id_e, 1.02)
# ttbare_wjetsMC.setParamEffect(reco_e, 1.02)
# ttbare_wjetsMC.autoMCStats()
addBtagSyst(background, "W_jets", "ttbare", ttbare_wjetsMC)

# addVJetsSyst(background, "W_jets", "ttbare", ttbare_wjetsMC)
pileupUp=template(background, 'W_jets','pileupUp','ttbare')[0] 
pileupDown=template(background, 'W_jets','pileupDown','ttbare')[0] 
ttbare_wjetsMC.setParamEffect(pileup, pileupUp, pileupDown)

ttbare.addSample(ttbare_wjetsMC)

# ###
# # top-antitop model
# ###

ttbare_ttTemplate = template(background, "TT", "nominal","ttbare", read_sumw2=True)
ttbare_ttMC = rl.TemplateSample(
    ch_name + "_ttMC",
    rl.Sample.BACKGROUND,
    ttbare_ttTemplate
)
ttbare_ttMC.setParamEffect(lumi, 1.027)
# ttbare_ttMC.setParamEffect(trig_e, 1.01)
# ttbare_ttMC.setParamEffect(veto_tau, 1.03)
# ttbare_ttMC.setParamEffect(jec, 1.05)
# ttbare_ttMC.setParamEffect(id_e, 1.02)
# ttbare_ttMC.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "TT", "ttbare", ttbare_ttMC)
# ttbare_ttMC.setParamEffect(ttMC_norm, 1.2)
ttbare.addSample(ttbare_ttMC)

# ###
# # Other MC-driven processes
# ###
# ST Background
ttbare_stTemplate = template(background, "ST", "nominal", "ttbare")
ttbare_st = rl.TemplateSample(
    ch_name + "_stMC", rl.Sample.BACKGROUND, ttbare_stTemplate
)
ttbare_st.setParamEffect(lumi, 1.027)
# ttbare_st.setParamEffect()
# ttbare_st.setParamEffect(trig_e, 1.01)
# ttbare_st.setParamEffect(veto_tau, 1.03)
# # ttbare_st.setParamEffect(st_norm, 1.2) something related to signal
# ttbare_st.setParamEffect(jec, 1.05)
# ttbare_st.setParamEffect(id_e, 1.02)
# ttbare_st.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "ST", "ttbare", ttbare_st)
trigeUp=template(background, 'ST', 'trig_eUp', 'ttbare')[0] 
trigeDown=template(background, 'ST', 'trig_eDown', 'ttbare')[0]

ttbare_st.setParamEffect(trig_e, trigeUp, trigeDown)
ttbare.addSample(ttbare_st)

# DY Jet Background
ttbare_dyjetsTemplate = template(background, "DY_jets", "nominal","ttbare")
ttbare_dyjets = rl.TemplateSample(
    ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, ttbare_dyjetsTemplate
)
ttbare_dyjets.setParamEffect(lumi, 1.027)
# ttbare_dyjets.setParamEffect(trig_e, 1.01)
# ttbare_dyjets.setParamEffect(veto_tau, 1.03)
# # ttbare_dyjets.setParamEffect(zjetsMC_norm, 1.4)
# ttbare_dyjets.setParamEffect(jec, 1.05)
# ttbare_dyjets.setParamEffect(id_e, 1.02)
# ttbare_dyjets.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "DY_jets", "ttbare", ttbare_dyjets)
addVJetsSyst(background,"DY_jets", "ttbare", ttbare_dyjets)
print("hi")
print(ttbare_dyjets.getParamEffect(ew1))
ttbare.addSample(ttbare_dyjets)


# V Jets Background
ttbare_vvTemplate = template(background, "VV", "nominal", "ttbare")
ttbare_vv = rl.TemplateSample(
    ch_name + "_vvMC", rl.Sample.BACKGROUND, ttbare_vvTemplate
)
ttbare_vv.setParamEffect(lumi, 1.027)
# ttbare_vv.setParamEffect(trig_e, 1.01)
# ttbare_vv.setParamEffect(veto_tau, 1.03)
# # ttbare_vv.setParamEffect(vv_norm, 1.2)
# ttbare_vv.setParamEffect(jec, 1.05)
# ttbare_vv.setParamEffect(id_e, 1.02)
# ttbare_vv.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "VV", "ttbare", ttbare_vv)
ttbare.addSample(ttbare_vv)

# QCD background
ttbare_qcdTemplate = template(background, "QCD", "nominal", "ttbare")
ttbare_qcd = rl.TemplateSample(
    ch_name + "_qcdMC", rl.Sample.BACKGROUND, ttbare_qcdTemplate
)
ttbare_qcd.setParamEffect(lumi, 1.027)
# ttbare_qcd.setParamEffect(trig_e, 1.01)
# ttbare_qcd.setParamEffect(veto_tau, 1.03)
# # ttbare_qcd.setParamEffect(qcde_norm, 2.0)
# ttbare_qcd.setParamEffect(jec, 1.05)
# ttbare_qcd.setParamEffect(id_e, 1.02)
# ttbare_qcd.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "QCD", "ttbare", ttbare_qcd)
ttbare.addSample(ttbare_qcd)


for s in signal['ttbare'].identifiers('process'):
    print(s)
    if 'Mphi1995_Mchi1000' not in str(s): continue
    ttbare_signalTemplate = template(signal,s,'nominal','ttbare')
    ttbare_signal=rl.TemplateSample(ch_name+'_'+str(s), rl.Sample.SIGNAL, ttbare_signalTemplate)
    ttbare_signal.setParamEffect(lumi, 1.027)
#     sr_signal.setParamEffect(trig_met, 1.01)
#     sr_signal.setParamEffect(veto_tau, 1.03)
#     sr_signal.setParamEffect(jec, 1.05)
    btagUp=template(signal, 'Mphi1995_Mchi1000','btagUp','ttbare')[0] 
    btagDown=template(signal, 'Mphi1995_Mchi1000','btagDown','ttbare')[0] 
    print(btagDown)
    ttbare_signal.setParamEffect(btag, btagUp, btagDown)
#     print(ttbare_signal.getParamEffect(btag))
    trigeUp=template(signal, 'Mphi1995_Mchi1000', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(signal, 'Mphi1995_Mchi1000', 'trig_eDown', 'ttbare')[0]
    print(trigeUp)
    ttbare_signal.setParamEffect(trig_e, trigeUp, trigeDown)
    ttbare.addSample(ttbare_signal)
    
    
    

[1.00290665 0.99441349 0.99157184 0.99483229 0.99364973 1.00519698
 0.99117453 0.14556905 1.18212365 0.9730222  0.9872623 ]
[0.98919835 0.98750973 0.98592611 0.98499902 0.98421368 0.98360965
 0.98299371 0.98222783 0.98145656 0.98040794 0.97924968]
[0.98497443 0.98433233 0.98397025 0.98415741 0.98221263 0.9799729
 0.98125066 0.98065254 0.98010007 0.97736429 0.97264407]
[0.97565126 1.03221988 0.98470451 0.98195523 0.98474391 1.0065767
 0.9781372  1.01105798 0.97735137 0.95952909 1.02484385]
1.000037490928439
hi
[1.00003749 1.00003749 1.00003749 1.00003749 1.00003749 1.00003749
 1.00003749 1.00003749 1.00003749 1.00003749 1.00003749]
[0.99061715 0.99444966 1.00328847 1.00347103 0.98123113 1.0345075
 1.1419427  0.97777272 0.97853621 1.         0.98775749]
[0.96835872 1.00977663 1.08375915 0.98517985 1.05780096 0.91331927
 1.00415155 1.02593308 0.95673186 0.96958558 0.99555886]
Mphi1995_Mchi1000
[1.00912537 1.00397977 1.01775038 0.96512069 1.00434641 1.00421368
 1.00605051 1.0022332  1.0027

In [ ]:
template(signal, 'Mphi1995_Mchi1000', 'trig_eUp', 'wjete')

In [35]:
signal['wjete'].integrate("process", "Mphi1995_Mchi1000").sum('npvs').integrate("systematic", 'trig_eUp').values()

{(): array([ 7.53661369, 16.23366055, 14.70197397, 13.23989184, 17.36249656,
        15.88282215, 15.97061372, 23.85615068, 32.1988018 , 21.49431074,
        55.94499508])}

In [10]:
model.name


'ControlRegionModel'

In [11]:
model.renderCombine("test_combine")



In [134]:
signal

{'sre': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9978>,
 'srm': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c99e8>,
 'ttbare': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9a20>,
 'ttbarm': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9a58>,
 'wjete': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9a90>,
 'wjetm': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9ac8>}

In [113]:
for s in signal['wjete'].identifiers('process'):
    print(s)
    if 'Mhs_50' not in str(s): continue
    wjete_signalTemplate = template(signal,s,'nominal','wjete')
    wjete_signal=rl.TemplateSample(ch_name+'_'+str(s), rl.Sample.SIGNAL, wjete_signalTemplate)
    wjete_signal.setParamEffect(lumi, 1.027)
#     sr_signal.setParamEffect(trig_met, 1.01)
#     sr_signal.setParamEffect(veto_tau, 1.03)
#     sr_signal.setParamEffect(jec, 1.05)
    btagUp=template(signal, s,'btagUp','wjete')[0]
    btagDown=template(signal, s,'btagDown','wjete')[0]
    wjete_signal.setParamEffect(btag, btagUp, btagDown)
    wjete.addSample(wjete_signal)
    


Mphi1995_Mchi1000
Mphi2000_Mchi1500
Mphi2000_Mchi500


In [114]:
def renderRoofit(self, workspace):
    import ROOT
    install_roofit_helpers()
    pdfName = self.name + '_simPdf'
    dataName = self.name + '_observation'
    rooSimul = workspace.pdf(pdfName)
    rooData = workspace.data(dataName)
    if rooSimul == None and rooData == None:  # noqa: E711
        channelCat = ROOT.RooCategory(self.name + '_channel', self.name + '_channel')
        # TODO s+b, b-only separate?
        rooSimul = ROOT.RooSimultaneous(self.name + '_simPdf', self.name + '_simPdf', channelCat)
        obsmap = ROOT.std.map('string, RooDataHist*')()
        for channel in self:
            channelCat.defineType(channel.name)
            pdf, obs = channel.renderRoofit(workspace)
            rooSimul.addPdf(pdf, channel.name)
            # const string magic: https://root.cern.ch/phpBB3/viewtopic.php?f=15&t=16882&start=15#p86985
            obsmap.insert(ROOT.std.pair('const string, RooDataHist*')(channel.name, obs))

        workspace.add(rooSimul)
        rooObservable = ROOT.RooArgList(channel.observable.renderRoofit(workspace))
        # that's right I don't need no CombDataSetFactory
        rooData = ROOT.RooDataHist(self.name + '_observation', 'Combined observation', rooObservable, channelCat, obsmap)
        workspace.add(rooData)
    elif rooSimul == None or rooData == None:  # noqa: E711
        raise RuntimeError('Model %r has a pdf or dataset already embedded in workspace %r' % (self, workspace))
    rooSimul = workspace.pdf(pdfName)
    rooData = workspace.data(dataName)
    return rooSimul, rooData

In [115]:
def renderCombine(self, h):
    import ROOT
    if not os.path.exists(outputPath):
        os.makedirs(outputPath)
    workspace = ROOT.RooWorkspace(self.name)
    self.renderRoofit(workspace)
    if workspace.writeToFile(os.path.join(outputPath, "%s.root" % self.name)):
        print(hi)
    for channel in self:
        channel.renderCard(os.path.join(outputPath, "%s.txt" % channel.name), self.name)
    with open(os.path.join(outputPath, "build.sh"), "w") as fout:
        cstr = " ".join("{0}={0}.txt".format(channel.name) for channel in self)
        fout.write("combineCards.py %s > %s_combined.txt\n" % (cstr, "model"))
        if self.t2w_config is None:
            fout.write("text2workspace.py model_combined.txt")
        else:
            fout.write("text2workspace.py {} model_combined.txt".format(self.t2w_config))

In [43]:
background['ttbare'].integrate('process', 'TT').integrate("systematic", 'btagUp').sum("npvs").values()

{(): array([9694.79798364, 6865.83507224, 3182.28106042, 2496.56073451,
        2451.00099807, 1729.79274506, 1189.27385461, 1136.97939632,
         796.65731001,  397.85451646,  392.42881793])}

In [44]:
background['ttbare'].integrate('process', 'TT').integrate("systematic", 'btagDown').sum("npvs").values()

{(): array([9694.79798364, 6865.83507224, 3182.28106042, 2496.56073451,
        2451.00099807, 1729.79274506, 1189.27385461, 1136.97939632,
         796.65731001,  397.85451646,  392.42881793])}

In [ ]:
workspace = ROOT.RooWorkspace(model.name)

In [ ]:
pdfName = model.name + '_simPdf'
rooSimul = workspace.pdf(pdfName)

In [ ]:
rooSimul

In [ ]:
model.renderRoofit('workspace')

In [ ]:
workspace.writeToFile()

In [ ]:
data_hists['template'].integrate('process').integrate('region', 'wjete').integrate('systematic').sum('npvs').sum('mT_corrected').values()

In [24]:
# # year='2018'
# # model_id = 'ControlRegionModel'
# # model = rl.Model(model_id)
# ch_name = "ttbare" 
# # Create a channel/Control region
# ttbare = rl.Channel(ch_name)
# # add the Channel/CR to the model
# model.addChannel(ttbare)

# ###
# # Add data distribution to the channel
# ###

# if year == "2018":
#     ttbare.setObservation(template(data, "EGamma", "data", "ttbare"))
# else:
#     ttbare.setObservation(template(data, "SingleElectron", "data","ttbare"))

# ###
# # W(->lnu)+jets data-driven model
# ###

# ttbare_wjetsTemplate = template(background, "W_jets", "nominal", "ttbare", read_sumw2=True)
# ttbare_wjetsMC = rl.TemplateSample(
#     ch_name + "_wjetsMC",
#     rl.Sample.BACKGROUND,
#     ttbare_wjetsTemplate
# )
# ttbare_wjetsMC.setParamEffect(lumi, 1.027)
# # ttbare_wjetsMC.setParamEffect(trig_e, 1.01)
# # ttbare_wjetsMC.setParamEffect(veto_tau, 1.03)
# # ttbare_wjetsMC.setParamEffect(wjets_norm, 1.4)
# # ttbare_wjetsMC.setParamEffect(jec, 1.05)
# # ttbare_wjetsMC.setParamEffect(id_e, 1.02)
# # ttbare_wjetsMC.setParamEffect(reco_e, 1.02)
# #ttbare_wjetsMC.autoMCStats()
# # addBtagSyst(background, "W_jets", "ttbare", ttbare_wjetsMC)
# # addVJetsSyst(background, "W+jets", "ttbare", ttbare_wjetsMC)

# # ttbare_wjetsTFstatParameters =  np.array([rl.NuisanceParameter("ttbare_"+year+"_wjetsTFstat_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i, "shape") for i in range(ttbare_wjetsTemplate[0].size)])
# # ttbare_wjetsTransferFactor = ttbare_wjetsMC.getExpectation() / sr_wjetsMC.getExpectation()
# # nominal =  ttbare_wjetsTemplate[0] / sr_wjetsTemplate[0]
# # dz = simple_error_propagation(sr_wjetsTemplate[0], ttbare_wjetsTemplate[0], sr_wjetsTemplate[3], ttbare_wjetsTemplate[3])
# # ttbare_wjetsTransferFactor = ttbare_wjetsTransferFactor * ( 1. + (dz/nominal)*ttbare_wjetsTFstatParameters )
# # ttbare_wjets = rl.TransferFactorSample(
# #     ch_name + "_wjets", rl.Sample.BACKGROUND, ttbare_wjetsTransferFactor, sr_wjets
# # )
# ttbare.addSample(ttbare_wjetsMC)

# # ###
# # # top-antitop model
# # ###

# ttbare_ttTemplate = template(background, "TT", "nominal","ttbare", read_sumw2=True)
# ttbare_ttMC = rl.TemplateSample(
#     ch_name + "_ttMC",
#     rl.Sample.BACKGROUND,
#     ttbare_ttTemplate
# )
# ttbare_ttMC.setParamEffect(lumi, 1.027)
# # ttbare_ttMC.setParamEffect(trig_e, 1.01)
# # ttbare_ttMC.setParamEffect(veto_tau, 1.03)
# # ttbare_ttMC.setParamEffect(jec, 1.05)
# # ttbare_ttMC.setParamEffect(id_e, 1.02)
# # ttbare_ttMC.setParamEffect(reco_e, 1.02)
# # addBtagSyst(background, "TT", "ttbare", ttbare_ttMC)
# # ttbare_ttMC.setParamEffect(ttMC_norm, 1.2)
# ttbare.addSample(ttbare_ttMC)

# # ###
# # # Other MC-driven processes
# # ###
# # ST Background
# ttbare_stTemplate = template(background, "ST", "nominal", "ttbare")
# ttbare_st = rl.TemplateSample(
#     ch_name + "_stMC", rl.Sample.BACKGROUND, ttbare_stTemplate
# )
# ttbare_st.setParamEffect(lumi, 1.027)
# # ttbare_st.setParamEffect(trig_e, 1.01)
# # ttbare_st.setParamEffect(veto_tau, 1.03)
# # # ttbare_st.setParamEffect(st_norm, 1.2) something related to signal
# # ttbare_st.setParamEffect(jec, 1.05)
# # ttbare_st.setParamEffect(id_e, 1.02)
# # ttbare_st.setParamEffect(reco_e, 1.02)
# # addBtagSyst(background, "ST", "ttbare", ttbare_st)
# ttbare.addSample(ttbare_st)

# # DY Jet Background
# ttbare_dyjetsTemplate = template(background, "DY_jets", "nominal","ttbare")
# ttbare_dyjets = rl.TemplateSample(
#     ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, ttbare_dyjetsTemplate
# )
# ttbare_dyjets.setParamEffect(lumi, 1.027)
# # ttbare_dyjets.setParamEffect(trig_e, 1.01)
# # ttbare_dyjets.setParamEffect(veto_tau, 1.03)
# # # ttbare_dyjets.setParamEffect(zjetsMC_norm, 1.4)
# # ttbare_dyjets.setParamEffect(jec, 1.05)
# # ttbare_dyjets.setParamEffect(id_e, 1.02)
# # ttbare_dyjets.setParamEffect(reco_e, 1.02)
# # addBtagSyst(background, "DY+jets", "ttbare", ttbare_dyjets)
# # addVJetsSyst(background,"DY+jets", "ttbare", ttbare_dyjets)
# ttbare.addSample(ttbare_dyjets)


# # V Jets Background
# ttbare_vvTemplate = template(background, "VV", "nominal", "ttbare")
# ttbare_vv = rl.TemplateSample(
#     ch_name + "_vvMC", rl.Sample.BACKGROUND, ttbare_vvTemplate
# )
# ttbare_vv.setParamEffect(lumi, 1.027)
# # ttbare_vv.setParamEffect(trig_e, 1.01)
# # ttbare_vv.setParamEffect(veto_tau, 1.03)
# # # ttbare_vv.setParamEffect(vv_norm, 1.2)
# # ttbare_vv.setParamEffect(jec, 1.05)
# # ttbare_vv.setParamEffect(id_e, 1.02)
# # ttbare_vv.setParamEffect(reco_e, 1.02)
# # addBtagSyst(background, "VV", "ttbare", ttbare_vv)
# ttbare.addSample(ttbare_vv)

# # QCD background
# ttbare_qcdTemplate = template(background, "QCD", "nominal", "ttbare")
# ttbare_qcd = rl.TemplateSample(
#     ch_name + "_qcdMC", rl.Sample.BACKGROUND, ttbare_qcdTemplate
# )
# ttbare_qcd.setParamEffect(lumi, 1.027)
# # ttbare_qcd.setParamEffect(trig_e, 1.01)
# # ttbare_qcd.setParamEffect(veto_tau, 1.03)
# # # ttbare_qcd.setParamEffect(qcde_norm, 2.0)
# # ttbare_qcd.setParamEffect(jec, 1.05)
# # ttbare_qcd.setParamEffect(id_e, 1.02)
# # ttbare_qcd.setParamEffect(reco_e, 1.02)
# addBtagSyst(background, "QCD", "ttbare", ttbare_qcd)
# ttbare.addSample(ttbare_qcd)

In [ ]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

mT_corrected_binning_dict = {
    "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
}



def remap_histograms(hists):
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY+jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W+jets"] = ("W+*",)
#     bkg_map["W+HF"] = ("W+HF",)
#     bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z_jets"] = ("Z+*",)
#     bkg_map["Z+HF"] = ("Z+HF",)
#     bkg_map["Z+LF"] = ("Z+LF",)
#     bkg_map["G+jets"] = ("G+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SingleMuon"] = ("SingleMuon",)
    data_map["EGamma"] = ("EGamma",)
    sig_map["Mphi1995_Mchi1000"] = ("Mphi-1995_Mchi-1000")
    sig_map["Mphi2000_Mchi1500"] = ("Mphi-2000_Mchi-1500")
    sig_map["Mphi2000_Mchi500"] = ("Mphi-2000_Mchi-500")
#     haven't processed signal yet
    for signal in hists['sig']['template'].identifiers('process'):
        if 'mchi' not in str(signal): continue
        sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
        signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial mT_corrected binning',bkg_hists["template"].axis("mT_corrected").edges())


    bkg_hists["template"] = bkg_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    signal_hists["template"] = signal_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    data_hists["template"] = data_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )

    hists = {"bkg": bkg_hists, "data": data_hists, "sig":signal_hists}

    return hists


def addBtagSyst(dictionary, process, region, templ):
    btagUp = template(dictionary, process, "btagUp", region)[0]
    btagDown = template(dictionary, process, "btagDown", region)[0]
    templ.setParamEffect(btag, btagUp, btagDown)

def addVJetsSyst(dictionary, process, region, templ):
    def addSyst(dictionary, process, region, templ, syst, string):
        histogram = dictionary[region].integrate("process", process)
#         print(histogram.integrate("systematic", "nominal").sum('npvs').values()[()])
        nominal=histogram.integrate("systematic", "nominal").sum('npvs').values()[()]
        up=histogram.integrate("systematic", string+"Up").sum('npvs').values()[()]
        down=histogram.integrate("systematic",string+"Down").sum('npvs').values()[()]
        systUp = np.array( up.sum() / nominal.sum() )
        systUp[np.isnan(systUp)] = 1.
        systUp = systUp.sum()
        templ.setParamEffect(syst, systUp)
    addSyst(dictionary, process, region, templ, ew1, "ew1")
    addSyst(dictionary, process, region, templ, ew2W, "ew2W")
    addSyst(dictionary, process, region, templ, ew2Z, "ew2Z")
    addSyst(dictionary, process, region, templ, ew3W, "ew3W")
    addSyst(dictionary, process, region, templ, ew3Z, "ew3Z")
    addSyst(dictionary, process, region, templ, mix, "mix")
    addSyst(dictionary, process, region, templ, qcd1, "qcd1")
    addSyst(dictionary, process, region, templ, qcd2, "qcd2")
    addSyst(dictionary, process, region, templ, qcd3, "qcd3")
    
def template(dictionary, process, systematic, region, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process).sum('npvs')
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    zerobins = nominal <= 0.
    output = nominal
    
    if "nominal" not in systematic and "data" not in systematic:
    #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("mT_corrected")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "mT_corrected", sumw2)
    return (output, binning, "mT_corrected")

def model(year):

#     model_id = year + category + "recoil" + str(recoil)
#     print(model_id)
#     model = rl.Model(str(s) + model_id)



    model_id = 'w'
    model = rl.Model(model_id)
    ch_name = "ttbare" 
    # Create a channel/Control region
    ttbare = rl.Channel(ch_name)
    # add the Channel/CR to the model
    model.addChannel(ttbare)
    
    ###
    # Add data distribution to the channel
    ###

    if year == "2018":
        ttbare.setObservation(template(data, "EGamma", "data", "ttbare"))
    else:
        ttbare.setObservation(template(data, "SingleElectron", "data","ttbare"))
        
    #template for signal
    for s in signal['ttbare'].identifiers('process'):
        print(s)
        if 'Mphi1995' not in str(s): continue
        ttbare_signalTemplate = template(signal,s,'nominal','ttbare')
        ttbare_signal=rl.TemplateSample(ch_name+'_'+str(s), rl.Sample.SIGNAL, ttbare_signalTemplate)
        ttbare_signal.setParamEffect(lumi, 1.027)
    #     sr_signal.setParamEffect(trig_met, 1.01)
    #     sr_signal.setParamEffect(veto_tau, 1.03)
    #     sr_signal.setParamEffect(jec, 1.05)
        #btag
        btagUp=template(signal, s,'btagUp','ttbare')[0]
        btagDown=template(signal, s,'btagDown','ttbare')[0]
        ttbare_signal.setParamEffect(btag, btagUp, btagDown)
        #trigger Electron
        trigeUp=template(signal, s, 'trig_eUp', 'ttbare')[0] 
        trigeDown=template(signal, s, 'trig_eDown', 'ttbare')[0]
        ttbare_signal.setParamEffect(trig_e, trigeUp, trigeDown)
        #id Electron
        id_eUp=template(signal, s, 'id_eUp', 'ttbare')[0] 
        id_eDown=template(signal, s, 'id_eDown', 'ttbare')[0]
        ttbare_signal.setParamEffect(id_e, id_eUp, id_eDown)
        #pileup
        pileupUp=template(signal, s,'pileupUp','ttbare')[0] 
        pileupDown=template(signal, s,'pileupDown','ttbare')[0] 
        ttbare_signal.setParamEffect(pileup, pileupUp, pileupDown)
        ttbare.addSample(ttbare_signal)
    
    ###
    # W(->lnu)+jets data-driven model
    ###

    ttbare_wjetsTemplate = template(background, "W+jets", "nominal", "ttbare", read_sumw2=True)
    ttbare_wjetsMC = rl.TemplateSample(
        ch_name + "_wjetsMC",
        rl.Sample.BACKGROUND,
        ttbare_wjetsTemplate
    )
    ttbare_wjetsMC.setParamEffect(lumi, 1.027)
#     ttbare_wjetsMC.setParamEffect(trig_e, 1.01)
#     ttbare_wjetsMC.setParamEffect(veto_tau, 1.03)
#     ttbare_wjetsMC.setParamEffect(wjets_norm, 1.4)
#     ttbare_wjetsMC.setParamEffect(jec, 1.05)
#     ttbare_wjetsMC.setParamEffect(id_e, 1.02)
#     ttbare_wjetsMC.setParamEffect(reco_e, 1.02)
    #ttbare_wjetsMC.autoMCStats()
    addBtagSyst(background, "W+jets", "ttbare", ttbare_wjetsMC)
    #trigger Electron
    trigeUp=template(background, 'W+jets', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'W+jets', 'trig_eDown', 'ttbare')[0]
    ttbare_wjetsMC.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "W+jets", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "W+jets", 'id_eDown', 'ttbare')[0]
    ttbare_wjetsMC.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'W+jets','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'W+jets','pileupDown','ttbare')[0] 
    ttbare_wjetsMC.setParamEffect(pileup, pileupUp, pileupDown)
    # addVJetsSyst(background, "W+jets", "ttbare", ttbare_wjetsMC)
    ttbare.addSample(ttbare_wjetsMC)

    # ###
    # # top-antitop model
    # ###

    ttbare_ttTemplate = template(background, "TT", "nominal","ttbare", read_sumw2=True)
    ttbare_ttMC = rl.TemplateSample(
        ch_name + "_ttMC",
        rl.Sample.BACKGROUND,
        ttbare_ttTemplate
    )
    ttbare_ttMC.setParamEffect(lumi, 1.027)
#     ttbare_ttMC.setParamEffect(trig_e, 1.01)
#     ttbare_ttMC.setParamEffect(veto_tau, 1.03)
#     ttbare_ttMC.setParamEffect(jec, 1.05)
#     ttbare_ttMC.setParamEffect(id_e, 1.02)
#     ttbare_ttMC.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "TT", "ttbare", ttbare_ttMC)
    #trigger Electron
    trigeUp=template(background, 'TT', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'TT', 'trig_eDown', 'ttbare')[0]
    ttbare_ttMC.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "TT", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "TT", 'id_eDown', 'ttbare')[0]
    ttbare_ttMC.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'TT','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'TT','pileupDown','ttbare')[0] 
    ttbare_ttMC.setParamEffect(pileup, pileupUp, pileupDown)
    # ttbare_ttMC.setParamEffect(ttMC_norm, 1.2)
    ttbare.addSample(ttbare_ttMC)

    # ###
    # # Other MC-driven processes
    # ###
    # ST Background
    ttbare_stTemplate = template(background, "ST", "nominal", "ttbare")
    ttbare_st = rl.TemplateSample(
        ch_name + "_stMC", rl.Sample.BACKGROUND, ttbare_stTemplate
    )
    ttbare_st.setParamEffect(lumi, 1.027)
#     ttbare_st.setParamEffect(trig_e, 1.01)
#     ttbare_st.setParamEffect(veto_tau, 1.03)
#     ttbare_st.setParamEffect(st_norm, 1.2) something related to signal
#     ttbare_st.setParamEffect(jec, 1.05)
#     ttbare_st.setParamEffect(id_e, 1.02)
#     ttbare_st.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "ST", "ttbare", ttbare_st)
    #trigger Electron
    trigeUp=template(background, 'ST', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'ST', 'trig_eDown', 'ttbare')[0]
    ttbare_st.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "ST", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "ST", 'id_eDown', 'ttbare')[0]
    ttbare_st.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'ST','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'ST','pileupDown','ttbare')[0] 
    ttbare_st.setParamEffect(pileup, pileupUp, pileupDown)
    ttbare.addSample(ttbare_st)

    # DY Jet Background
    ttbare_dyjetsTemplate = template(background, "DY+jets", "nominal","ttbare")
    ttbare_dyjets = rl.TemplateSample(
        ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, ttbare_dyjetsTemplate
    )
    ttbare_dyjets.setParamEffect(lumi, 1.027)
#     ttbare_dyjets.setParamEffect(trig_e, 1.01)
#     ttbare_dyjets.setParamEffect(veto_tau, 1.03)
#     ttbare_dyjets.setParamEffect(zjetsMC_norm, 1.4)
#     ttbare_dyjets.setParamEffect(jec, 1.05)
#     ttbare_dyjets.setParamEffect(id_e, 1.02)
#     ttbare_dyjets.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "DY+jets", "ttbare", ttbare_dyjets)
#     addVJetsSyst(background,"DY+jets", "ttbare", ttbare_dyjets)
    #trigger Electron
    trigeUp=template(background, 'DY+jets', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'DY+jets', 'trig_eDown', 'ttbare')[0]
    ttbare_dyjets.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "DY+jets", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "DY+jets", 'id_eDown', 'ttbare')[0]
    ttbare_dyjets.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'DY+jets','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'DY+jets','pileupDown','ttbare')[0] 
    ttbare_dyjets.setParamEffect(pileup, pileupUp, pileupDown)
    ttbare.addSample(ttbare_dyjets)


    # V Jets Background
    ttbare_vvTemplate = template(background, "VV", "nominal", "ttbare")
    ttbare_vv = rl.TemplateSample(
        ch_name + "_vvMC", rl.Sample.BACKGROUND, ttbare_vvTemplate
    )
    ttbare_vv.setParamEffect(lumi, 1.027)
#     ttbare_vv.setParamEffect(trig_e, 1.01)
#     ttbare_vv.setParamEffect(veto_tau, 1.03)
#     ttbare_vv.setParamEffect(vv_norm, 1.2)
#     ttbare_vv.setParamEffect(jec, 1.05)
#     ttbare_vv.setParamEffect(id_e, 1.02)
#     ttbare_vv.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "VV", "ttbare", ttbare_vv)
    #trigger Electron
    trigeUp=template(background, 'VV', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'VV', 'trig_eDown', 'ttbare')[0]
    ttbare_vv.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "VV", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "VV", 'id_eDown', 'ttbare')[0]
    ttbare_vv.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'VV','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'VV','pileupDown','ttbare')[0] 
    ttbare_vv.setParamEffect(pileup, pileupUp, pileupDown)
    ttbare.addSample(ttbare_vv)

    # QCD background
    ttbare_qcdTemplate = template(background, "QCD", "nominal", "ttbare")
    ttbare_qcd = rl.TemplateSample(
        ch_name + "_qcdMC", rl.Sample.BACKGROUND, ttbare_qcdTemplate
    )
    ttbare_qcd.setParamEffect(lumi, 1.027)
#     ttbare_qcd.setParamEffect(trig_e, 1.01)
#     ttbare_qcd.setParamEffect(veto_tau, 1.03)
#     ttbare_qcd.setParamEffect(qcde_norm, 2.0)
#     ttbare_qcd.setParamEffect(jec, 1.05)
#     ttbare_qcd.setParamEffect(id_e, 1.02)
#     ttbare_qcd.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "QCD", "ttbare", ttbare_qcd)
    #trigger Electron
    trigeUp=template(background, 'QCD', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'QCD', 'trig_eDown', 'ttbare')[0]
    ttbare_qcd.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "QCD", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "QCD", 'id_eDown', 'ttbare')[0]
    ttbare_qcd.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'QCD','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'QCD','pileupDown','ttbare')[0] 
    ttbare_qcd.setParamEffect(pileup, pileupUp, pileupDown)
    ttbare.addSample(ttbare_qcd)
    return model

if __name__ == "__main__":
    if not os.path.exists("datacards"):
        os.mkdir("datacards")
    parser = OptionParser()
    parser.add_option("-y", "--year", help="year", dest="year", default="")
    (options, args) = parser.parse_args()
    year = options.year
    year = '2018'

    mT_corrected_binning_dict = {
        "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
        "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
        "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
    }
    mT_corrected_binning = mT_corrected_binning_dict[year]
    mT_correctedbins = np.array(mT_corrected_binning)
    nmT_corrected = len(mT_correctedbins) - 1

    ###
    # Extract histograms from input file
    ###

    print("Grouping histograms")
    hists = load("hists/UL_lep_monotop_v1_test_" + year + ".scaled")
    hists = remap_histograms(hists)
    data_hists = hists["data"]
    bkg_hists = hists["bkg"]
    signal_hists = hists["sig"]

    ###
    # Preparing histograms for fit
    ##

    data = {}
    for r in data_hists["template"].identifiers("region"):
        data[str(r)] = data_hists["template"].integrate("region", r)

    background = {}
    for r in bkg_hists["template"].identifiers("region"):
        background[str(r)] = bkg_hists["template"].integrate("region", r)

    signal = {}
    for r in signal_hists["template"].identifiers("region"):
        signal[str(r)] = signal_hists["template"].integrate("region", r)
    ###
    ###
    # Setting up systematics
    ###
    ###
    lumi = rl.NuisanceParameter("lumi" + year, "lnN")
    zjets_norm = rl.NuisanceParameter("zjets_norm", "lnN")
    wjets_norm = rl.NuisanceParameter("wjets_norm", "lnN")
    tt_norm = rl.NuisanceParameter("tt_norm", "lnN")
    id_e = rl.NuisanceParameter("id_e" + year, "shape")
    id_mu = rl.NuisanceParameter("id_mu" + year, "lnN")
    id_pho = rl.NuisanceParameter("id_pho" + year, "lnN")
    reco_e = rl.NuisanceParameter("reco_e" + year, "lnN")
    iso_mu = rl.NuisanceParameter("iso_mu" + year, "lnN")
    trig_e = rl.NuisanceParameter("trig_e" + year, "shape")
    trig_met = rl.NuisanceParameter("trig_met" + year, "lnN")
    trig_pho = rl.NuisanceParameter("trig_pho" + year, "lnN")
    veto_tau = rl.NuisanceParameter("veto_tau" + year, "lnN")
    jec = rl.NuisanceParameter("jec" + year, "lnN")
    #btag = rl.NuisanceParameter("btag" + year, "shapeN")  # AK4 btag
    btag = rl.NuisanceParameter("btag" + year, "shape")  # AK4 btag
    ew1 = rl.NuisanceParameter("ew1", "lnN")
    #ew2G = rl.NuisanceParameter("ew2G", "lnN")
    ew2W = rl.NuisanceParameter("ew2W", "lnN")
    ew2Z = rl.NuisanceParameter("ew2Z", "lnN")
    #ew3G = rl.NuisanceParameter("ew3G", "lnN")
    ew3W = rl.NuisanceParameter("ew3W", "lnN")
    ew3Z = rl.NuisanceParameter("ew3Z", "lnN")
    mix = rl.NuisanceParameter("mix", "lnN")
    #muF = rl.NuisanceParameter("muF", "lnN")
    #muR = rl.NuisanceParameter("muR", "lnN")
    qcd1 = rl.NuisanceParameter("qcd1", "lnN")
    qcd2 = rl.NuisanceParameter("qcd2", "lnN")
    qcd3 = rl.NuisanceParameter("qcd3", "lnN")
    whf_fraction = rl.NuisanceParameter("whf_fraction", "lnN")
    zhf_fraction = rl.NuisanceParameter("zhf_fraction", "lnN")
    #ghf_fraction = rl.NuisanceParameter("ghf_fraction", "shapeN")
    pileup = rl.NuisanceParameter("pileup"+year, "shape")
    
    with open(
        "data/"+"test"+year+".model",
        "wb",) as fout:
        pickle.dump(model(year), fout, protocol=2)